# Mandelbrot Fractal – Parallel & Dask Implementation
**Hand-in Report**

This notebook covers:
1. NumPy vectorized baseline
2. Parallel multiprocessing – chunk size analysis & speedup
3. Dask local multi-core – chunk size tuning & comparison to NumPy
4. Dask distributed cluster execution

## 0. Imports & Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
from multiprocessing import cpu_count

# Core shared functions
from mandelbrot_core import (
    build_complex_grid,
    mandelbrot_numpy,
    mandelbrot_block,
    mandelbrot_chunk_worker,
    timed,
    plot_mandelbrot,
    plot_chunk_analysis,
    plot_speedup,
    plot_dask_comparison,
)

# Parallel runner
from mandelbrot_multiprocessing import mandelbrot_parallel

# Dask runner
from mandelbrot_dask import mandelbrot_dask

print(f'CPUs available: {cpu_count()}')

# Global settings
WIDTH    = 1000
HEIGHT   = 1000
MAX_ITER = 100
REPEATS  = 3   # how many times to repeat each benchmark

---
## 1. NumPy Vectorized Baseline

This is the reference implementation. All speedup values are relative to this.

In [ ]:
t_numpy, M_numpy = timed(mandelbrot_numpy, WIDTH, HEIGHT,
                          max_iter=MAX_ITER, repeats=REPEATS)
print(f'NumPy baseline (mean of {REPEATS} runs): {t_numpy:.3f}s')

ModuleNotFoundError: No module named 'mandelbrot_core'

In [ ]:
plot_mandelbrot(M_numpy, title='Mandelbrot Set – NumPy baseline')

---
## 2. Parallel Multiprocessing

### 2A. Chunk Size Analysis

We sweep over different chunk sizes (rows per worker task) and different numbers of processes.
The optimal chunk size balances:
- **too small** → too many tasks, high scheduling/communication overhead  
- **too large** → workers become unbalanced (load imbalance)

In [ ]:
from multiprocessing import Pool

chunk_sizes    = [10, 25, 50, 100, 200, 500]
process_counts = [1, 2, 4, min(8, cpu_count())]

rows = []
for P in process_counts:
    for cs in chunk_sizes:
        times = []
        for _ in range(REPEATS):
            t0 = time.perf_counter()
            mandelbrot_parallel(WIDTH, HEIGHT, P, cs, max_iter=MAX_ITER)
            times.append(time.perf_counter() - t0)
        avg = float(np.mean(times))
        rows.append({'processes': P, 'chunk_size': cs, 'time': avg})
        print(f'P={P:2d}, chunk={cs:4d}  →  {avg:.3f}s')

df_chunks = pd.DataFrame(rows)
df_chunks.to_csv('chunk_size_results.csv', index=False)

In [ ]:
plot_chunk_analysis(df_chunks, filename='chunk_size_analysis.png')

In [ ]:
# Best chunk size = lowest average time across all P values
best_cs = int(df_chunks.groupby('chunk_size')['time'].mean().idxmin())
print(f'Optimal chunk size: {best_cs} rows')

**Discussion:** *(fill in after running)*  
Explain why the optimal chunk size is what it is. Consider: IPC overhead for small chunks, load imbalance for large chunks.

### 2B. Speedup vs Number of Processes

In [ ]:
speedup_rows = []
for P in process_counts:
    times = []
    for _ in range(REPEATS):
        t0 = time.perf_counter()
        mandelbrot_parallel(WIDTH, HEIGHT, P, best_cs, max_iter=MAX_ITER)
        times.append(time.perf_counter() - t0)
    avg     = float(np.mean(times))
    speedup = t_numpy / avg
    speedup_rows.append({'processes': P, 'time': avg, 'speedup': speedup})
    print(f'P={P:2d}  →  {avg:.3f}s   speedup={speedup:.2f}×')

df_speedup = pd.DataFrame(speedup_rows)
df_speedup.to_csv('speedup_results.csv', index=False)

In [ ]:
plot_speedup(df_speedup, filename='speedup_analysis.png')

**Discussion:** *(fill in after running)*  
- Is speedup linear? Why or why not? (Amdahl's Law)  
- At what P does speedup saturate?  
- Is multiprocessing faster than the NumPy baseline? Why might it not be?

---
## 3. Dask – Local Multi-core

### 3A. Chunk Size Tuning (Note 1)

For local Dask runs, chunk size that fits in the L2 cache tends to perform best.

In [ ]:
import dask.array as da

dask_chunk_sizes = [32, 64, 128, 250, 500, 1000]
dask_rows = []

for cs in dask_chunk_sizes:
    times = []
    for _ in range(REPEATS):
        t0 = time.perf_counter()
        mandelbrot_dask(WIDTH, HEIGHT, chunk_size=cs, max_iter=MAX_ITER)
        times.append(time.perf_counter() - t0)
    avg = float(np.mean(times))
    dask_rows.append({'chunk_size': cs, 'time': avg})
    print(f'chunk={cs:5d}  →  {avg:.3f}s')

df_dask_chunks = pd.DataFrame(dask_rows)
df_dask_chunks.to_csv('dask_chunk_size_results.csv', index=False)

best_dask_cs = int(df_dask_chunks.loc[df_dask_chunks['time'].idxmin(), 'chunk_size'])
print(f'\nBest Dask chunk size: {best_dask_cs}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(df_dask_chunks['chunk_size'], df_dask_chunks['time'],
        marker='o', color='darkorange')
ax.set_xlabel('Chunk Size')
ax.set_ylabel('Mean Time (s)')
ax.set_title('Dask Local – Chunk Size vs Execution Time')
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig('dask_chunk_sweep.png', dpi=150)
plt.show()

### 3B. Dask vs NumPy Across Problem Sizes

In [ ]:
sizes = [500, 1000, 2000]
compare_rows = []

for sz in sizes:
    t_np, _ = timed(mandelbrot_numpy, sz, sz, max_iter=MAX_ITER, repeats=REPEATS)
    t_dk, _ = timed(mandelbrot_dask,  sz, sz,
                    chunk_size=best_dask_cs, max_iter=MAX_ITER, repeats=REPEATS)
    speedup  = t_np / t_dk
    compare_rows.append({'size': sz, 'numpy_time': t_np,
                         'dask_time': t_dk, 'speedup': speedup})
    print(f'{sz}×{sz}  NumPy={t_np:.3f}s  Dask={t_dk:.3f}s  speedup={speedup:.2f}×')

df_compare = pd.DataFrame(compare_rows)
df_compare.to_csv('dask_vs_numpy_results.csv', index=False)

In [ ]:
plot_dask_comparison(df_compare, filename='dask_vs_numpy.png')

**Discussion:** *(fill in after running)*  
- Does Dask outperform NumPy? At which problem sizes?  
- Why does the advantage grow (or shrink) with problem size?  
- How does the map_blocks approach help vs. feeding the grid directly?

---
## 4. Dask Distributed – Cluster Execution

Set `SCHEDULER_ADDRESS = None` for a local simulated cluster,  
or set it to `'tcp://<strato-ip>:8786'` for the real Strato cluster.

In [ ]:
from dask.distributed import Client, LocalCluster

SCHEDULER_ADDRESS = None   # ← change to 'tcp://<ip>:8786' for real cluster

if SCHEDULER_ADDRESS is None:
    cluster = LocalCluster(n_workers=4, threads_per_worker=1)
    client  = Client(cluster)
    mode    = 'LocalCluster (4 workers)'
else:
    cluster = None
    client  = Client(SCHEDULER_ADDRESS)
    mode    = f'Strato cluster @ {SCHEDULER_ADDRESS}'

print(f'Mode     : {mode}')
print(f'Dashboard: {client.dashboard_link}')
print(f'Workers  : {len(client.scheduler_info()["workers"])}')

In [ ]:
DIST_WIDTH  = 2000
DIST_HEIGHT = 2000

dist_times = []
for run in range(REPEATS):
    t0 = time.perf_counter()
    mandelbrot_dask(DIST_WIDTH, DIST_HEIGHT,
                    chunk_size=best_dask_cs, max_iter=MAX_ITER)
    elapsed = time.perf_counter() - t0
    dist_times.append(elapsed)
    print(f'Run {run+1}: {elapsed:.3f}s')

dist_avg = float(np.mean(dist_times))

# Compare to local NumPy for same size
t_np_large, _ = timed(mandelbrot_numpy, DIST_WIDTH, DIST_HEIGHT,
                       max_iter=MAX_ITER, repeats=REPEATS)

print(f'\nDistributed Dask ({mode}): {dist_avg:.3f}s')
print(f'NumPy baseline (same size): {t_np_large:.3f}s')
print(f'Speedup: {t_np_large/dist_avg:.2f}×')

In [ ]:
# Summary bar chart: NumPy vs Dask local vs Dask distributed
labels = ['NumPy\nbaseline', f'Dask local\n(chunk={best_dask_cs})', f'Dask\n{mode}']
times  = [t_np_large,
           timed(mandelbrot_dask, DIST_WIDTH, DIST_HEIGHT,
                 chunk_size=best_dask_cs, max_iter=MAX_ITER, repeats=REPEATS)[0],
           dist_avg]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(labels, times, color=['steelblue', 'darkorange', 'green'])
ax.bar_label(bars, fmt='%.2fs', padding=3)
ax.set_ylabel('Mean Execution Time (s)')
ax.set_title(f'Execution Time Comparison ({DIST_WIDTH}×{DIST_HEIGHT}, max_iter={MAX_ITER})')
ax.grid(True, axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig('final_comparison.png', dpi=150)
plt.show()

In [ ]:
client.close()
if cluster is not None:
    cluster.close()
print('Cluster shut down.')

**Discussion:** *(fill in after running)*  
- How does distributed Dask compare to local Dask and NumPy?  
- What is the network overhead cost when running on a real cluster?  
- For the distributed case, what chunk size would you recommend and why?

---
## 5. Summary Table

*(Update values after all experiments are done)*

In [ ]:
summary = pd.DataFrame([
    {'Implementation': 'NumPy vectorized',       'Time (s)': t_numpy,   'Speedup': 1.0},
    {'Implementation': f'Parallel (P=4, chunk={best_cs})',
                                                  'Time (s)': df_speedup[df_speedup['processes']==4]['time'].values[0],
                                                  'Speedup':  df_speedup[df_speedup['processes']==4]['speedup'].values[0]},
    {'Implementation': f'Dask local (chunk={best_dask_cs})',
                                                  'Time (s)': df_compare[df_compare['size']==WIDTH]['dask_time'].values[0],
                                                  'Speedup':  df_compare[df_compare['size']==WIDTH]['speedup'].values[0]},
])
summary['Time (s)']  = summary['Time (s)'].round(3)
summary['Speedup']   = summary['Speedup'].round(2)
print(summary.to_string(index=False))
summary.to_csv('summary_results.csv', index=False)

---
## 6. Conclusion

*(Write your conclusion here after running all experiments)*

- Which implementation is fastest and why?
- How does problem size affect the benefit of parallelism?
- What are the bottlenecks (GIL, IPC overhead, network, scheduling overhead)?
- What chunk size strategy would you recommend for each scenario?